# Lib


In [ ]:
from pathlib import Path
import sys
import gc
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

PROJECT_ROOT = Path("../../").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from evaluation.checkpoints import build_and_load_model
from evaluation.metrics import evaluate_model

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

# Configuration


In [ ]:
SELECTED_METRIC = "dice"
METRICS = [
    "dice",
    "iou",
    "precision",
    "recall",
    "hd95",
]
DATA_DIR = PROJECT_ROOT / "data/data"
TEST_CSV = DATA_DIR / "sample/test.csv"
CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "image_height": 256,
    "image_width": 256,
    "num_classes": 3,
    "normalize_input": True,
    "input_mean": [0.485, 0.456, 0.406],
    "input_std": [0.229, 0.224, 0.225],
    "mean_teacher_state": "student",
    "batch_size": 16,
    "num_workers": 2,
}
CLASS_NAMES = ["background", "non-neoplastic", "neoplastic"]
METHODS = ["Supervised", "Mean Teacher", "Mean Teacher + Contrastive"]
COLORS = dict(zip(METHODS, ["#4477AA", "#EE7733", "#228833"]))
ARCHITECTURES = ["UNet", "SwinUNet"]
SPLITS = ["quarter", "half"]
MODEL_META = {
    "unet": ("UNet", METHODS[0]),
    "mean_teacher_unet": ("UNet", METHODS[1]),
    "contrastive_unet": ("UNet", METHODS[2]),
    "swin": ("SwinUNet", METHODS[0]),
    "mean_teacher_swin": ("SwinUNet", METHODS[1]),
    "contrastive_swin": ("SwinUNet", METHODS[2]),
}
print("Device:", CONFIG["device"])
print("Primary metric: Dice")

In [ ]:
CHECKPOINTS = {
    "unet": {
        "half": PROJECT_ROOT / "models/baseline/best_UNet_half.pth",
        "quarter": PROJECT_ROOT / "models/baseline/best_UNet_quarter.pth",
    },
    "mean_teacher_unet": {
        "half": PROJECT_ROOT / "models/mean teacher/best_mean_teacher_unet_half.pth",
        "quarter": PROJECT_ROOT
        / "models/mean teacher/best_mean_teacher_UNet_quarter.pth",
    },
    "contrastive_unet": {
        "half": None,
        "quarter": PROJECT_ROOT / "models/contrastive/best_ContrastiveUNet_quarter.pth",
    },
}

SWIN_CHECKPOINTS = {
    "swin": {
        "half": PROJECT_ROOT / "models/baseline/best_SwinUNet_half.pth",
        "quarter": PROJECT_ROOT / "models/baseline/best_SwinUNet_quarter.pth",
    },
    "mean_teacher_swin": {
        "half": PROJECT_ROOT
        / "models/mean teacher/best_mean_teacher_SwinUNet_half.pth",
        "quarter": PROJECT_ROOT
        / "models/mean teacher/best_mean_teacher_SwinUNet_quarter.pth",
    },
    "contrastive_swin": {
        "half": PROJECT_ROOT / "models/contrastive/best_ContrastiveSwinUNet_half.pth",
        "quarter": PROJECT_ROOT
        / "models/contrastive/best_ContrastiveSwinUNet_quarter.pth",
    },
}
CHECKPOINTS.update(SWIN_CHECKPOINTS)

# Checkpoint availability


In [ ]:
records = []
for model_type, runs in CHECKPOINTS.items():
    architecture, method = MODEL_META[model_type]
    for split, path in runs.items():
        available = path is not None and Path(path).is_file()
        records.append(
            dict(
                model=f"{model_type}/{split}",
                model_type=model_type,
                architecture=architecture,
                method=method,
                split=split,
                path=path,
                status="Available" if available else "Missing checkpoint",
            )
        )
manifest = pd.DataFrame(records)
display(manifest[["architecture", "method", "split", "status"]])


# Test data and preprocessing


In [ ]:
def decode_ground_truth(mask_rgb):
    mask = np.asarray(mask_rgb)

    if mask.ndim == 2:
        unique_values = np.unique(mask)

        if np.isin(unique_values, [0, 1, 2]).all():
            return mask.astype(np.uint8)

        raise ValueError(
            "Grayscale mask must contain only class IDs 0, 1, 2. "
            f"Found: {unique_values[:20]}"
        )

    if mask.ndim != 3 or mask.shape[-1] != 3:
        raise ValueError(f"Expected HxWx3 RGB mask, got shape={mask.shape}")

    colors = np.array(
        [
            [0, 0, 0],
            [0, 255, 0],
            [255, 0, 0],
        ],
        dtype=np.float32,
    )

    distances = np.sum(
        (mask.astype(np.float32)[:, :, None, :] - colors[None, None, :, :]) ** 2,
        axis=-1,
    )

    return np.argmin(distances, axis=-1).astype(np.uint8)


def pick_column(
    dataframe,
    explicit_column=None,
    candidates=None,
    required=True,
):
    if explicit_column is not None:
        if explicit_column not in dataframe.columns:
            raise KeyError(
                f"Column '{explicit_column}' not found. "
                f"Available: {dataframe.columns.tolist()}"
            )
        return explicit_column

    normalized = {str(column).strip().lower(): column for column in dataframe.columns}

    for candidate in candidates or []:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    if len(dataframe.columns) == 1:
        return dataframe.columns[0]

    if required:
        raise KeyError(
            "Cannot infer CSV column automatically. "
            f"Available: {dataframe.columns.tolist()}"
        )

    return None


class SegmentationEvaluationDataset(Dataset):
    IMAGE_COLUMN_CANDIDATES = [
        "image_path",
        "image",
        "image_name",
        "filename",
        "file_name",
        "file",
        "id",
    ]

    MASK_COLUMN_CANDIDATES = [
        "mask",
        "mask_name",
        "mask_path",
        "label",
        "label_path",
        "gt",
        "gt_path",
    ]

    def __init__(
        self,
        dataframe,
        image_dir,
        mask_dir,
        transform=None,
        image_col=None,
        mask_col=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform

        self.image_col = pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )

        self.mask_col = pick_column(
            self.df,
            explicit_column=mask_col,
            candidates=self.MASK_COLUMN_CANDIDATES,
            required=False,
        )

    @staticmethod
    def resolve_path(root_dir, value):
        path = Path(str(value))
        return path if path.is_absolute() else Path(root_dir) / path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image_path = self.resolve_path(
            self.image_dir,
            row[self.image_col],
        )

        mask_value = (
            row[self.mask_col] if self.mask_col is not None else row[self.image_col]
        )

        mask_path = self.resolve_path(
            self.mask_dir,
            mask_value,
        )

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )
        mask_raw = cv2.imread(
            str(mask_path),
            cv2.IMREAD_UNCHANGED,
        )

        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")

        image = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(
                mask_raw,
                cv2.COLOR_BGR2RGB,
            )

        mask = decode_ground_truth(mask_raw)

        if self.transform is not None:
            transformed = self.transform(
                image=image,
                mask=mask,
            )
            image = transformed["image"]
            mask = transformed["mask"]

        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask)

        return {
            "image": image,
            "mask": mask.long(),
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_name": image_path.name,
        }


def build_eval_transform(config):
    transform_steps = [
        A.Resize(
            height=int(config["image_height"]),
            width=int(config["image_width"]),
        )
    ]

    if config["normalize_input"]:
        transform_steps.append(
            A.Normalize(
                mean=tuple(config["input_mean"]),
                std=tuple(config["input_std"]),
            )
        )

    transform_steps.append(ToTensorV2())

    return A.Compose(transform_steps)

In [ ]:
test_df = pd.read_csv(TEST_CSV)
if test_df.empty:
    raise ValueError("The test set is empty.")
if test_df["image_path"].duplicated().any():
    raise ValueError("Duplicate test images would bias the comparison.")
for column in ["image_path", "mask_path"]:
    missing = [
        str(DATA_DIR / str(value))
        for value in test_df[column]
        if not (DATA_DIR / str(value)).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"{column}: missing {len(missing)} files; first: {missing[0]}"
        )
print(f"Test images: {len(test_df)} - identical for every checkpoint")

# Evaluate every available checkpoint


In [ ]:
tables = []
model_details = []
confusion_matrices = {}
for entry in manifest.itertuples(index=False):
    if entry.status != "Available":
        print(f"Skipped: {entry.model} - {entry.status}")
        continue
    print(f"Evaluating {entry.model}", flush=True)
    model, config, info = build_and_load_model(entry.path, entry.model_type, CONFIG)
    if (
        int(config["image_height"]),
        int(config["image_width"]),
        int(config["num_classes"]),
    ) != (256, 256, 3):
        raise ValueError(
            f"{entry.model}: expected 256 x 256 and 3 classes for this comparison."
        )
    dataset = SegmentationEvaluationDataset(
        test_df,
        DATA_DIR,
        DATA_DIR,
        transform=build_eval_transform(config),
        image_col="image_path",
        mask_col="mask_path",
    )
    loader = DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=CONFIG["device"] == "cuda",
    )
    result, matrix = evaluate_model(
        model,
        entry.model,
        loader,
        CONFIG["device"],
        num_classes=3,
        class_names=CLASS_NAMES,
        compute_hd95=True,
    )
    for key in ["architecture", "method", "split"]:
        result[key] = getattr(entry, key)
    result["samples"] = len(dataset)
    result["weights"] = info["weights"]
    tables.append(result)
    confusion_matrices[entry.model] = matrix
    model_details.append(dict(model=entry.model, **info))
    print(
        f'Completed {entry.model}: Dice = {result.loc[result.class_id.eq("macro_all"), "dice"].iloc[0]:.4f}',
        flush=True,
    )
    del model, loader, dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
if not tables:
    raise ValueError("No checkpoints are available. Update CHECKPOINTS.")
evaluation_df = pd.concat(tables, ignore_index=True)


# Primary comparison - Dice


In [ ]:
class_rows = evaluation_df[evaluation_df.class_id.isin([0, 1, 2])].copy()
summary = evaluation_df[evaluation_df.class_id.eq("macro_all")].copy()
for label, group in class_rows.groupby("model"):
    if (
        set(group.class_id) != {0, 1, 2}
        or len(group) != 3
        or not np.isfinite(group.dice).all()
    ):
        raise ValueError(
            f"{label}: Dice comparison requires three defined class scores."
        )
    summary.loc[summary.model.eq(label), "dice"] = group.dice.sum() / 3
summary["dice_rank"] = (
    summary.groupby("split").dice.rank(method="min", ascending=False).astype(int)
)
display(
    summary.sort_values(["split", "dice_rank"])[
        ["split", "dice_rank", "architecture", "method", "dice", "samples", "weights"]
    ].round(4)
)


In [ ]:
def finish_figure(fig):
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def grouped_bars(metric):
    fig, axes = plt.subplots(
        1, len(SPLITS), figsize=(14, 5), squeeze=False, sharey=True
    )
    x = np.arange(len(ARCHITECTURES))
    width = 0.24
    for ax, split in zip(axes.flat, SPLITS):
        subset = summary[summary.split.eq(split)]
        for index, method in enumerate(METHODS):
            scores = (
                subset[subset.method.eq(method)]
                .set_index("architecture")[metric]
                .reindex(ARCHITECTURES)
            )
            bars = ax.bar(
                x + (index - 1) * width,
                scores,
                width,
                color=COLORS[method],
                label=method,
            )
            ax.bar_label(
                bars,
                labels=[f"{v:.3f}" if pd.notna(v) else "" for v in scores],
                fontsize=8,
                padding=3,
            )
            for position, value in zip(x + (index - 1) * width, scores):
                if pd.isna(value):
                    ax.text(position, 0, "N/A", ha="center", va="bottom", fontsize=8)
        ax.set_xticks(x, ARCHITECTURES)
        ax.set_title(f"{split.title()} labeled training data")
        ax.set_ylabel(metric.upper())
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        if metric != "hd95":
            ax.set_ylim(0, 1.1)
        else:
            ax.set_ylim(bottom=0)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(
        f"{metric.upper()} comparison"
        + (" - lower is better" if metric == "hd95" else "")
    )
    finish_figure(fig)


grouped_bars("dice")

# Select a metric


In [ ]:
def metric_charts(metric):
    if metric not in METRICS:
        raise ValueError(f"Choose one of {METRICS}")
    grouped_bars(metric)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    for ax, architecture in zip(axes, ARCHITECTURES):
        subset = summary[summary.architecture.eq(architecture)]
        for method in METHODS:
            scores = (
                subset[subset.method.eq(method)]
                .set_index("split")[metric]
                .reindex(SPLITS)
            )
            ax.plot([25, 50], scores, marker="o", color=COLORS[method], label=method)
        ax.set_title(architecture)
        ax.set_xticks([25, 50], ["25% (quarter)", "50% (half)"])
        ax.set_xlabel("Labeled training fraction")
        ax.set_ylabel(metric.upper())
        ax.grid(alpha=0.2)
        if metric != "hd95":
            ax.set_ylim(0, 1.02)
    axes[0].legend(fontsize=8)
    finish_figure(fig)

    heat = class_rows.pivot(index="model", columns="class_name", values=metric).reindex(
        columns=CLASS_NAMES
    )
    heat["Mean"] = summary.set_index("model")[metric]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.45 * len(heat))))
    values = heat.to_numpy(dtype=float)
    cmap = plt.get_cmap("viridis_r" if metric == "hd95" else "viridis").copy()
    cmap.set_bad("#dddddd")
    im = ax.imshow(
        np.ma.masked_invalid(values),
        aspect="auto",
        cmap=cmap,
        vmin=0,
        vmax=None if metric == "hd95" else 1,
    )
    ax.set_xticks(range(len(heat.columns)), heat.columns, rotation=15, ha="right")
    ax.set_yticks(range(len(heat)), heat.index)
    for row in range(len(heat)):
        for col in range(len(heat.columns)):
            value = values[row, col]
            ax.text(
                col,
                row,
                f"{value:.3f}" if np.isfinite(value) else "N/A",
                ha="center",
                va="center",
                color="black",
                bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1),
            )
    ax.set_title(metric.upper() + " - Per Class Comparison")
    fig.colorbar(im, ax=ax)
    finish_figure(fig)


metric_charts(SELECTED_METRIC)

# Metric relationships


In [ ]:
def correlation_charts(selected_metric):
    correlation = summary[METRICS].corr(min_periods=3)
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(
        np.ma.masked_invalid(correlation.to_numpy()), vmin=-1, vmax=1, cmap="coolwarm"
    )
    ax.set_xticks(range(len(METRICS)), METRICS, rotation=40, ha="right")
    ax.set_yticks(range(len(METRICS)), METRICS)
    for i in range(len(METRICS)):
        for j in range(len(METRICS)):
            v = correlation.iloc[i, j]
            ax.text(
                j,
                i,
                f"{v:.2f}" if pd.notna(v) else "N/A",
                ha="center",
                va="center",
                fontsize=8,
            )
    ax.set_title(f"Metric correlation")
    fig.colorbar(im, ax=ax, label="Pearson r")
    finish_figure(fig)

    other = "iou" if selected_metric == "dice" else selected_metric
    fig, ax = plt.subplots(figsize=(15, 6))
    for architecture, marker in zip(ARCHITECTURES, ["o", "^"]):
        for method in METHODS:
            rows = summary[
                summary.architecture.eq(architecture) & summary.method.eq(method)
            ]
            ax.scatter(
                rows.dice,
                rows[other],
                color=COLORS[method],
                marker=marker,
                s=65,
                label=f"{architecture} - {method}",
            )
            for row in rows.itertuples():
                ax.annotate(
                    row.split,
                    (row.dice, getattr(row, other)),
                    xytext=(4, 5),
                    textcoords="offset points",
                    fontsize=8,
                )
    ax.set_xlabel("Dice")
    ax.set_ylabel(other.upper())
    ax.set_title(f"Dice {other.upper()}")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8, loc="best")
    finish_figure(fig)


correlation_charts(SELECTED_METRIC)

# Interactive metric selector


In [ ]:
if widgets is not None:
    selector = widgets.Dropdown(
        options=METRICS, value=SELECTED_METRIC, description="Metric:"
    )
    chart_output = widgets.Output()

    def update_metric(change):
        with chart_output:
            chart_output.clear_output(wait=True)
            metric_charts(change["new"])
            correlation_charts(change["new"])

    selector.observe(update_metric, names="value")
    display(selector, chart_output)
else:
    print(
        "Set SELECTED_METRIC in Configuration, then rerun metric_charts and correlation_charts."
    )